<a href="https://colab.research.google.com/github/suryaph971/Langchain/blob/main/streamlit_frontend_question_answering_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [3]:
!pip install -q chromadb
!pip install -q docx2txt
!pip install -q pypdf
!pip install -q streamlit
#!pip install -q  iktoken
!pip install -q langchain_google_genai
!pip install -q langchain

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement iktoken (from versions: none)
ERROR: No matching distribution found for iktoken
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [4]:
!pip install -q  tiktoken

In [6]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.4 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [6]:
import streamlit as st
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import Chroma

def load_document(file):
    import os
    name,extension=os.path.splitext(file)
    if extension=='.pdf':
        from langchain.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader=PyPDFLoader(file)
    elif extension=='.docx':
        from langchain.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader=Docx2txtLoader(file)
    elif extension=='.txt':
        from langchain.document_loaders import TextLoader
        print(f'Loading {file}')
        loader=TextLoader(file)
    else:
        print('Document format is not supported!')
        return None
    data = loader.load()
    return data


In [2]:
def chunk_data(data,chunk_size=256,chunk_overlap=20):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(data)
    return chunks

In [3]:
def create_embeddings(chunks):
    embeddings=GoogleGenerativeAIEmbeddings(model='text-embedding-004',dimensions=1536)
    vector_store=Chroma.from_documents(chunks,embedding=embeddings,persist_directory='db')
    return vector_store

In [8]:
def ask_and_get_answer(vector_store,q,k=3):
    from langchain.chains import RetrievalQA
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash',temperature=1.0)
    retriever=vector_store.as_retriever()
    chain=RetrievalQA.from_chain_type(llm=llm,chain_type='stuff',retriever=retriever)
    answer = chain.invoke(q)
    return answer['result']


In [5]:
def calculate_embedding_costs(texts):
    import tiktoken
    enc =tiktoken.encoding_for_model('text-embedding-004')
    total_tokens=sum([len(enc.encode(page.page_content)) for page in texts])
    return total_tokens,total_tokens/1000*0.0004

In [7]:
def clear_history():
    if 'history' in st.session_state:
        del st.session_state['history']

In [9]:
if __name__ == "__main__":
    import os
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(), override=True)
    st.subheader('LLM Question-Answering Application 🤖')
    with st.sidebar:
        uploaded_file = st.file_uploader('Upload a file:', type=['pdf', 'docx', 'txt'])
        chunk_size = st.number_input('Chunk size:', min_value=100, max_value=2048, value=512,on_change=clear_history)
        k=st.number_input('k',min_value=1,max_value=25,value=4,on_change=clear_history)
        add_data = st.button('Add Data',on_click=clear_history)
        if uploaded_file  and add_data:
            with st.spinner('Reading, chunking and embedding file ...'):
                bytes_data = uploaded_file.read()
                file_name=os.path.join('./',uploaded_file.name)
                with open(file_name,'wb') as f:
                    f.write(bytes_data)

                data=load_document(file_name)
                chunks=chunk_data(data,chunk_size=chunk_size)
                st.write(f'Chunk size: {chunk_size}, Chunks: {len(chunks)}')
                tokens, embedding_cost = calculate_embedding_cost(chunks)
                st.write(f'Embedding cost: ${embedding_cost:.4f}')
                vector_store = create_embeddings(chunks)
                st.session_state.vs = vector_store
                st.success('File uploaded, chunked and embedded successfully.')
    q=st.text_input('Ask a question about the content of your file:')
    if q:
        standard_answer = "Answer only based on the text you received as input. Don't search external sources. " \
                          "If you can't answer then return `I DONT KNOW`."
        q = f"{q} {standard_answer}"
        if 'vs' in st.session_state:
            vector_store = st.session_state.vs
            st.write(f'k: {k}')
            answer = ask_and_get_answer(vector_store, q,k)
            st.write(answer)
            st.text_area('LLM Answer: ', value=answer)
            st.divider()
            if 'history' not in st.session_state:
                st.session_state.history = ''
            value = f'Q: {q} \nA: {answer}'
            st.session_state.history = f'{value} \n {"-" * 100} \n {st.session_state.history}'
            h = st.session_state.history
            st.text_area(label='Chat History', value=h, key='history', height=400)




2025-09-11 16:16:50.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-11 16:16:50.591 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-11 16:16:50.595 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-11 16:16:50.596 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-11 16:16:50.598 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-11 16:16:50.600 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-11 16:16:50.602 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-11 16:16:50.604 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar